<a href="https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

i use logistic reggression as well as random tree as my methodes. This fits my
lane because both methodes output probability which i can rank.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
dim_content_path = f"{rel}/dim_content.parquet"
fact_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

base = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions) AS impressions_90d,
        SUM(f.gsc_clicks) AS clicks_90d,
        AVG(f.gsc_avg_position) AS avg_position,
        c.word_count,
        c.content_created_date,
        c.content_updated_date
    FROM read_parquet('{fact_path}') f
    JOIN read_parquet('{dim_content_path}') c USING (content_hash_id)
    GROUP BY 1, 2, c.word_count, c.content_created_date, c.content_updated_date
""").df()

base["ctr"] = base["clicks_90d"] / base["impressions_90d"].replace(0, pd.NA)
base["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(base["content_updated_date"])).dt.days
base = base.dropna(subset=["impressions_90d", "clicks_90d", "avg_position", "word_count", "days_since_update"])
base = base.reset_index(drop=True)

base["target"] = ((base["days_since_update"] >= 180) & (base["impressions_90d"] >= 500)).astype(int)
print(base["target"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

target
0    121418
1         5
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
because random split would count the user twice inflating the score

In [ ]:
X = base[["impressions_90d", "clicks_90d", "avg_position", "word_count", "days_since_update"]]
y = base["target"]
groups = base["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", base["client_hash_id"].iloc[train_idx].nunique())
print("Test clients:", base["client_hash_id"].iloc[test_idx].nunique())
print("Overlap:", set(base["client_hash_id"].iloc[train_idx]) & set(base["client_hash_id"].iloc[test_idx]))

Train clients: 32
Test clients: 15
Overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_probs = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

baseline_score_test = base.iloc[test_idx]["impressions_90d"] / base["impressions_90d"].max() + \
                       base.iloc[test_idx]["days_since_update"] / base["days_since_update"].max()
baseline_score_test = baseline_score_test / baseline_score_test.max()

results = pd.DataFrame({
    "method": ["baseline rule", "logistic regression", "random forest"],
    "roc_auc": [
        roc_auc_score(y_test, baseline_score_test),
        roc_auc_score(y_test, logreg_probs),
        roc_auc_score(y_test, rf_probs),
    ],
    "average_precision": [
        average_precision_score(y_test, baseline_score_test),
        average_precision_score(y_test, logreg_probs),
        average_precision_score(y_test, rf_probs),
    ]
})
results

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_r

,method,roc_auc,average_precision
0,baseline rule,NaN,0.0
1,logistic regression,NaN,0.0
2,random forest,NaN,0.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

test_df = base.iloc[test_idx].copy()
test_df["rf_prob"] = rf_probs
false_positives = test_df[(test_df["target"] == 0) & (test_df["rf_prob"] > 0.7)]
false_negatives = test_df[(test_df["target"] == 1) & (test_df["rf_prob"] < 0.3)]
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))
false_positives.head()

days_since_update    0.379630
impressions_90d      0.350746
clicks_90d           0.095799
avg_position         0.091507
word_count           0.082319
dtype: float64
False positives: 0
False negatives: 0


,content_hash_id,client_hash_id,impressions_90d,clicks_90d,avg_position,word_count,content_created_date,content_updated_date,ctr,days_since_update,target,rf_prob


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.